# Visualize occurrence of keywords in the GLOBALISE corpus over time

The below code does a count of search terms per inventory number using the Glocolli API, which exposes the Elastic search index of the GLOBALISE corpus. It then uses metadata from the archival description of the National Archives (specifically, the 'year_begin' value) to plot the occurrences over time. Note that the 'year_begin' value relates to the earliest known date of a document in a specific inventory number. This will not in all cases correspond to the date on which a keyword was mentioned.

In [ ]:
import requests
from collections import Counter
import time
import pandas as pd
import matplotlib.pyplot as plt

# API settings
base_url = "https://gloccoli.tt.di.huc.knaw.nl/projects/globalise/search"
index_name = "docs-2024-03-18"
query = {
    "text": "ADD SEARCH TERM(S) HERE",
    "terms": {}
}

# Pagination settings
page_size = 500
start = 0
total = None
invnr_counter = Counter()

while True:
    print(f"Fetching results {start} to {start + page_size}")
    url = f"{base_url}?indexName={index_name}&from={start}&size={page_size}"
    response = requests.post(url, json=query)
    data = response.json()

    if total is None:
        total = data["total"]["value"]
        print(f"Total results: {total}")

    results = data.get("results", [])
    if not results:
        break

    # Count invNr values
    for hit in results:
        invnr = hit.get("invNr")
        if invnr:
            invnr_counter[invnr] += 1

    start += page_size
    time.sleep(0.3)  # gentle rate limit

    if start >= total:
        break

# Make sure the keys in invnr_counter are str:
invnr_counter = {str(k): v for k, v in invnr_counter.items()}

# Load the CSV with date information of VOC inventory numbers and ensure inventory_number is treated as string
url = "https://raw.githubusercontent.com/globalise-huygens/globalise-visualizations/refs/heads/main/data/1.04.02_dates.csv"
df = pd.read_csv(url, sep="\t", dtype={"inventory_number": str})  # Replace with your path

# Add result counts by matching inventory_number to invnr_counter
df["result_count"] = df["inventory_number"].map(invnr_counter).fillna(0).astype(int)

# Group by year_begin and sum the counts
yearly_counts = df.groupby("year_begin")["result_count"].sum().reset_index()

# Plot the results
plt.figure(figsize=(12, 6))
plt.bar(yearly_counts["year_begin"], yearly_counts["result_count"], color='steelblue')
plt.xlabel("Year (year_begin)")
plt.ylabel("Number of search results")
plt.title("Search result count per inventory start year")
plt.xticks(rotation=45)
plt.tight_layout()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

